In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# extract unique HashFunctions
functions = {}
for filepath in sorted(Path("../experiment/data/azurefunctions-dataset2019/").glob("invocations*.csv")):
    print(filepath)
    df = pd.read_csv(filepath)
    d = filepath.stem.split(".")[-1]
    for function in df.HashFunction.unique():
        if function in functions.keys():
            functions[function].append(d)
        else:
            functions[function] = [d]

In [ ]:
# extract functions that are appearing in all the files
functions_of_interest = []

for key, value in functions.items():
    if len(value) == 14:
        functions_of_interest.append(key)

print(f"{len(functions_of_interest)} over {len(functions)} have been invoked all the 14 days")

In [ ]:
# Extract time series up to 2880 values (2 days)

folder = Path("../experiment/data/azurefunctions-dataset2019/")
from tqdm import tqdm

# construct time series
timeseries = {}
value_cols = [str(i) for i in range(1, 1441)]

dfs = []

for filepath in sorted(Path("../experiment/data/azurefunctions-dataset2019/").glob("invocations*.csv")):
    print(filepath)
    df = pd.read_csv(filepath)
    d = filepath.stem.split(".")[-1]

    df = df[df["HashFunction"].isin(functions_of_interest)]

    result_df = df.melt(id_vars = ["HashOwner", "HashApp", "HashFunction", "Trigger"], value_vars = [str(i) for i in range(1, 1441)], var_name="minute")
    
    # Convert minute to integer
    result_df["minute"] = result_df["minute"].astype(int)

    # Create 10-minute bins: minute 1-10 -> bin 0, minute 11-20 -> bin 1, etc.
    result_df["10_minute_bin"] = (result_df["minute"] - 1) // 10  # subtract 1 so bin 0 covers minutes 1–10

    # Group by HashFunction and bin, then sum
    grouped = result_df.groupby(["HashOwner", "HashApp", "HashFunction", "Trigger", "10_minute_bin"], as_index=False)["value"].sum()

    grouped["day"] = d
    dfs.append(grouped.copy())


In [ ]:
final = pd.concat(dfs)
final

In [ ]:

final['dt#'] = final.groupby('HashFunction').cumcount()
final

In [ ]:
final.to_csv("azure_10min_all.csv", index=False)

In [ ]:
final.head(10)

In [ ]:
len(final.HashFunction.unique())

In [ ]:
import numpy as np
var_dict = {}
std_dict = {}
zeros_dict = {}
len_unique = {}
for function, function_df in tqdm(final.groupby("HashFunction")):
    ts = function_df.sort_values(by="dt#").value.values
    # var_dict[function] = ts.var()
    # zeros_dict[function] = (ts == 0).sum()
    len_unique[function] = len(np.unique(ts))
    std_dict[function] = np.std(ts) / np.mean(ts)

# var_dict

In [ ]:
functions_to_select = [k for k, v in len_unique.items() if v > 1041]
functions_to_select

In [ ]:
vals = [v for k, v in std_dict.items() if k in functions_to_select]
vals

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(vals)


np.quantile(vals, np.arange(0.1, 1, .01))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(vals)
plt.xlim(0, 50)


np.quantile(vals, np.arange(0.1, 1, .01))


In [ ]:
forecasting_dataset = final[final["HashFunction"].isin(functions_to_select)]
forecasting_dataset = forecasting_dataset.rename(columns={"HashFunction": "ts", "value": "y"})[["ts", "dt#", "y"]].sort_values(by=["ts", "dt#"])

forecasting_dataset.to_csv("azure10min.csv", index=False)

In [ ]:
forecasting_dataset

In [ ]:
for function in functions_to_select[:20]:
    plt.plot(forecasting_dataset[forecasting_dataset["ts"] == function].sort_values(by='dt#').y.values)
    plt.show()

In [ ]:
len(functions_to_select)